In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 · Trust & Safety Evaluation
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Evaluar reglas heurísticas Trust & Safety sobre las features
# MAGIC previamente calculadas.
# MAGIC
# MAGIC Este notebook:
# MAGIC
# MAGIC - NO matricula reglas.
# MAGIC - NO calcula features.
# MAGIC - NO consolida riesgo.
# MAGIC - NO confirma fraude.
# MAGIC
# MAGIC ## Catálogo de reglas
# MAGIC
# MAGIC `workspace.yelp_gov.evaluation_rule`
# MAGIC
# MAGIC `workspace.yelp_gov.evaluation_rule_version`
# MAGIC
# MAGIC ## Entrada
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_features`
# MAGIC
# MAGIC ## Salida
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_evaluation`
# MAGIC
# MAGIC ## Grano
# MAGIC
# MAGIC **1 Review × 1 Rule Version × 1 Evaluation Run**
# MAGIC
# MAGIC ## Flujo
# MAGIC
# MAGIC Feature Engineering
# MAGIC → Rule Engine
# MAGIC → Review Evaluation
# MAGIC → Risk Consolidation


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F

import uuid
import json

from datetime import (
    datetime,
    timezone
)


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOV_SCHEMA = "yelp_gov"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"


FEATURE_TABLE = (
    f"{SILVER}.slv_review_features"
)

EVALUATION_TABLE = (
    f"{SILVER}.slv_review_evaluation"
)

RULE_TABLE = (
    f"{GOV}.evaluation_rule"
)

RULE_VERSION_TABLE = (
    f"{GOV}.evaluation_rule_version"
)

RUN_TABLE = (
    f"{GOV}.evaluation_run"
)

DQ_RESULT_TABLE = (
    f"{GOV}.dq_resultados"
)


PIPELINE_NAME = (
    "03_trust_safety_evaluacion"
)

PIPELINE_VERSION = "3.0.0"


# ------------------------------------------------------------
# Desarrollo
#
# False = ejecución completa
# True  = limita únicamente la evaluación.
#
# Las features ya fueron calculadas sobre el dataset completo.
# ------------------------------------------------------------

DEV_MODE = False
DEV_LIMIT = 50000


evaluation_run_id = str(
    uuid.uuid4()
)


evaluation_started_at = (
    datetime.now(
        timezone.utc
    )
)


print("=" * 72)
print("YELPCONNECT - TRUST & SAFETY EVALUATION")
print("=" * 72)

print(
    f"Evaluation Run : {evaluation_run_id}"
)

print(
    f"Pipeline       : {PIPELINE_NAME}"
)

print(
    f"Version        : {PIPELINE_VERSION}"
)

print(
    f"DEV MODE       : {DEV_MODE}"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # 3. Validaciones de entrada


# COMMAND ----------

# ============================================================
# 3. VALIDAR TABLAS
# ============================================================

required_tables = [

    FEATURE_TABLE,
    RULE_TABLE,
    RULE_VERSION_TABLE,
    DQ_RESULT_TABLE
]


for table_name in required_tables:

    if not spark.catalog.tableExists(
        table_name
    ):

        raise Exception(
            f"""
            ❌ Tabla requerida no encontrada:

            {table_name}
            """
        )


print(
    "✔ Tablas requeridas disponibles."
)


# COMMAND ----------

# ============================================================
# 4. VALIDAR QUALITY GATE SILVER
# ============================================================

latest_dq = (

    spark.table(
        DQ_RESULT_TABLE
    )

    .filter(
        F.col(
            "capa"
        )
        ==
        "silver"
    )

    .orderBy(
        F.desc(
            "ejecucion_ts"
        )
    )

    .select(
        "ejecucion_id"
    )

    .first()
)


if latest_dq is None:

    raise Exception(
        """
        ❌ No existe una ejecución DQ Silver.

        Trust & Safety requiere que el
        Quality Gate Silver se ejecute primero.
        """
    )


latest_dq_id = (
    latest_dq.ejecucion_id
)


critical_failures = (

    spark.table(
        DQ_RESULT_TABLE
    )

    .filter(
        F.col(
            "ejecucion_id"
        )
        ==
        latest_dq_id
    )

    .filter(
        F.lower(
            F.col(
                "criticidad"
            )
        )
        ==
        "alta"
    )

    .filter(
        F.col(
            "paso"
        )
        ==
        False
    )

    .count()
)


if critical_failures > 0:

    raise Exception(
        f"""
        ❌ Quality Gate Silver bloqueado.

        Fallas críticas:
        {critical_failures}
        """
    )


print(
    "✔ Quality Gate Silver superado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 5. Obtener reglas activas del catálogo
# MAGIC
# MAGIC Solo se ejecutan reglas que cumplan:
# MAGIC
# MAGIC `active_flag = true`
# MAGIC
# MAGIC y
# MAGIC
# MAGIC `status = ACTIVE`
# MAGIC
# MAGIC Por lo tanto R007 permanece fuera del motor
# MAGIC mientras continúe como DRAFT.


# COMMAND ----------

# ============================================================
# 5. REGLAS ACTIVAS
# ============================================================

active_rules_df = (

    spark.table(
        RULE_TABLE
    )
    .alias("r")

    .filter(
        F.col(
            "r.active_flag"
        )
        ==
        True
    )

    .join(

        spark.table(
            RULE_VERSION_TABLE
        )
        .alias("v")

        .filter(
            F.col(
                "v.status"
            )
            ==
            "ACTIVE"
        ),

        F.col(
            "r.rule_id"
        )
        ==
        F.col(
            "v.rule_id"
        ),

        "inner"
    )

    .select(

        F.col(
            "r.rule_id"
        ),

        F.col(
            "r.rule_name"
        ),

        F.col(
            "v.rule_version_id"
        ),

        F.col(
            "v.version"
        ),

        F.col(
            "v.parameters_json"
        ),

        F.col(
            "v.threshold"
        ),

        F.col(
            "v.weight"
        )
    )
)


# COMMAND ----------

# ============================================================
# 6. VALIDAR UNA SOLA VERSIÓN ACTIVE POR REGLA
# ============================================================

duplicate_active_versions = (

    active_rules_df

    .groupBy(
        "rule_id"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        !=
        1
    )

    .count()
)


if duplicate_active_versions > 0:

    raise Exception(
        """
        ❌ Existen reglas con cero o múltiples
        versiones ACTIVE.

        Cada regla activa debe tener exactamente
        una versión ACTIVE.
        """
    )


active_rule_rows = (
    active_rules_df.collect()
)


if len(
    active_rule_rows
) == 0:

    raise Exception(
        """
        ❌ No existen reglas Trust & Safety activas.
        """
    )


print(
    f"✔ Reglas ACTIVE: "
    f"{len(active_rule_rows)}"
)


# COMMAND ----------

# ============================================================
# 7. REGLAS SOPORTADAS POR ESTA VERSIÓN DEL MOTOR
#
# Si mañana R007 pasa a ACTIVE pero todavía no hemos
# implementado su expresión, el pipeline debe fallar.
#
# No debemos ignorarla silenciosamente.
# ============================================================

SUPPORTED_RULE_IDS = {

    "R001",
    "R002",
    "R003",
    "R004",
    "R005",
    "R006"
}


active_rule_ids = {
    row["rule_id"]
    for row in active_rule_rows
}


unsupported_rules = (
    active_rule_ids
    -
    SUPPORTED_RULE_IDS
)


if unsupported_rules:

    raise Exception(
        f"""
        ❌ Existen reglas ACTIVE que esta versión
        del motor todavía no implementa:

        {sorted(unsupported_rules)}

        Actualizar primero:
        03_trust_safety_evaluacion
        """
    )


print(
    "✔ Todas las reglas ACTIVE son soportadas."
)


# COMMAND ----------

# ============================================================
# 8. CONVERTIR CATÁLOGO A CONFIGURACIÓN
# ============================================================

rule_configs = {}


for row in active_rule_rows:

    parameters_raw = (
        row["parameters_json"]
        or
        "{}"
    )

    try:

        parameters = (
            json.loads(
                parameters_raw
            )
        )

    except Exception as exc:

        raise Exception(
            f"""
            ❌ parameters_json inválido:

            Rule:
            {row['rule_id']}

            JSON:
            {parameters_raw}
            """
        ) from exc


    rule_configs[
        row["rule_id"]
    ] = {

        "rule_version_id":
            row["rule_version_id"],

        "version":
            row["version"],

        "threshold":
            float(
                row["threshold"]
            )
            if row["threshold"] is not None
            else None,

        "weight":
            float(
                row["weight"]
            )
            if row["weight"] is not None
            else 0.0,

        "parameters":
            parameters
    }


# COMMAND ----------

# ============================================================
# 9. MOSTRAR CONFIGURACIÓN ACTIVA
# ============================================================

display(

    active_rules_df

    .orderBy(
        "rule_id"
    )
)


# COMMAND ----------

# ============================================================
# 10. VALIDAR PESOS
# ============================================================

active_weight = sum(

    config["weight"]

    for config
    in rule_configs.values()
)


print(
    f"Peso total ACTIVE: "
    f"{active_weight:.4f}"
)


if abs(
    active_weight - 1.0
) > 0.0001:

    print(
        "⚠ ADVERTENCIA: "
        "los pesos ACTIVE no suman 1.0. "
        "Revisar thresholds del Risk Consolidation."
    )

else:

    print(
        "✔ Pesos ACTIVE suman 1.0."
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 11. Validar features necesarias


# COMMAND ----------

# ============================================================
# 11. FEATURE REQUIREMENTS POR REGLA
# ============================================================

required_features_by_rule = {

    "R001": {
        "user_reviews_30m"
    },

    "R002": {
        "rating_deviation",
        "business_stars"
    },

    "R003": {
        "text_length",
        "text_repeat_count"
    },

    "R004": {
        "account_age_days",
        "user_review_count",
        "yelping_since"
    },

    "R005": {
        "user_business_review_count",
        "user_business_concentration"
    },

    "R006": {
        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_spike_ratio"
    }
}


feature_table_columns = set(

    spark.table(
        FEATURE_TABLE
    ).columns
)


missing_features = []


for rule_id in sorted(
    active_rule_ids
):

    required_columns = (
        required_features_by_rule[
            rule_id
        ]
    )

    missing = (
        required_columns
        -
        feature_table_columns
    )

    for column_name in sorted(
        missing
    ):

        missing_features.append(
            (
                rule_id,
                column_name
            )
        )


if missing_features:

    raise Exception(
        f"""
        ❌ Faltan features requeridas:

        {missing_features}

        Ejecutar o actualizar:
        04_feature_engineering_ts
        """
    )


print(
    "✔ Features requeridas disponibles."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 12. Crear tabla de ejecución


# COMMAND ----------

# ============================================================
# 12. EVALUATION RUN
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {RUN_TABLE}
    (
        evaluation_run_id   STRING,
        pipeline_name       STRING,
        pipeline_version    STRING,
        dataset_version     STRING,
        started_at          TIMESTAMP,
        finished_at         TIMESTAMP,
        execution_status    STRING
    )
    USING DELTA
    """
)


# COMMAND ----------

# ============================================================
# 13. LEER FEATURES
#
# Serverless:
# NO usamos cache() / persist().
# ============================================================

features = (

    spark.table(
        FEATURE_TABLE
    )
)


if DEV_MODE:

    features = (

        features

        .orderBy(
            "review_id"
        )

        .limit(
            DEV_LIMIT
        )
    )


review_count = (
    features.count()
)


print(
    f"Reviews a evaluar: "
    f"{review_count:,}"
)


# COMMAND ----------

# ============================================================
# 14. FEATURE VERSION
#
# Ahora dataset_version referencia directamente
# la ejecución de Feature Engineering.
# ============================================================

feature_version_rows = (

    features

    .select(
        "feature_execution_id"
    )

    .where(
        F.col(
            "feature_execution_id"
        ).isNotNull()
    )

    .distinct()

    .limit(2)

    .collect()
)


if len(
    feature_version_rows
) == 1:

    dataset_version = (
        feature_version_rows[0]
        .feature_execution_id
    )

elif len(
    feature_version_rows
) == 0:

    dataset_version = "UNKNOWN"

else:

    dataset_version = "MULTIPLE_FEATURE_EXECUTIONS"

    print(
        "⚠ La tabla de features contiene "
        "más de un feature_execution_id."
    )


print(
    f"Feature Version : "
    f"{dataset_version}"
)


# COMMAND ----------

# ============================================================
# 15. REGISTRAR RUN
# ============================================================

run_schema = """
evaluation_run_id STRING,
pipeline_name STRING,
pipeline_version STRING,
dataset_version STRING,
started_at TIMESTAMP,
finished_at TIMESTAMP,
execution_status STRING
"""


run_df = spark.createDataFrame(

    [
        (
            evaluation_run_id,
            PIPELINE_NAME,
            PIPELINE_VERSION,
            dataset_version,
            evaluation_started_at,
            None,
            "RUNNING"
        )
    ],

    run_schema
)


(
    run_df

    .write

    .format("delta")

    .mode("append")

    .saveAsTable(
        RUN_TABLE
    )
)


print(
    "✔ Evaluation Run registrado."
)


# COMMAND ----------

# ============================================================
# 16. FUNCIÓN ACTUALIZAR RUN
# ============================================================

def update_run_status(
    status
):

    spark.sql(
        f"""
        UPDATE
            {RUN_TABLE}

        SET
            execution_status =
                '{status}',

            finished_at =
                current_timestamp()

        WHERE
            evaluation_run_id =
                '{evaluation_run_id}'
        """
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 17. Motor de reglas
# MAGIC
# MAGIC Thresholds, versiones y pesos provienen del catálogo.
# MAGIC
# MAGIC La expresión técnica de cada regla permanece
# MAGIC implementada en este motor.


# COMMAND ----------

# ============================================================
# 17. CONSTRUCTOR DE REGLAS
# ============================================================

def build_rule_struct(
    rule_id
):

    config = (
        rule_configs[
            rule_id
        ]
    )

    params = (
        config["parameters"]
    )

    weight = (
        config["weight"]
    )

    rule_version_id = (
        config[
            "rule_version_id"
        ]
    )


    # ========================================================
    # R001 · Burst Activity
    # ========================================================

    if rule_id == "R001":

        window_minutes = int(

            params.get(
                "window_minutes",
                30
            )
        )


        if window_minutes != 30:

            raise Exception(
                f"""
                ❌ R001 solicita una ventana de
                {window_minutes} minutos.

                La Feature actual disponible es:
                user_reviews_30m.

                Actualizar primero Feature Engineering.
                """
            )


        min_reviews = int(

            params.get(
                "min_reviews",
                config["threshold"]
            )
        )


        evaluable = (

            F.col(
                "user_reviews_30m"
            ).isNotNull()
        )


        signal = (

            F.col(
                "user_reviews_30m"
            )
            >=
            min_reviews
        )


        reason = F.concat(

            F.lit(
                "Reviews usuario en 30m: "
            ),

            F.col(
                "user_reviews_30m"
            ).cast(
                "string"
            ),

            F.lit(
                f" | threshold: {min_reviews}"
            )
        )


    # ========================================================
    # R002 · Rating Deviation
    # ========================================================

    elif rule_id == "R002":

        rating_difference = float(

            params.get(
                "rating_difference",
                config["threshold"]
            )
        )


        evaluable = (

            F.col(
                "business_stars"
            ).isNotNull()

            &

            F.col(
                "rating_deviation"
            ).isNotNull()
        )


        signal = (

            F.col(
                "rating_deviation"
            )
            >=
            rating_difference
        )


        reason = F.concat(

            F.lit(
                "Desviación rating: "
            ),

            F.round(
                F.col(
                    "rating_deviation"
                ),
                2
            ).cast(
                "string"
            ),

            F.lit(
                f" | threshold: "
                f"{rating_difference}"
            )
        )


    # ========================================================
    # R003 · Repeated Text
    # ========================================================

    elif rule_id == "R003":

        min_repetitions = int(

            params.get(
                "min_repetitions",
                config["threshold"]
            )
        )


        min_text_length = int(

            params.get(
                "min_text_length",
                40
            )
        )


        evaluable = (

            F.col(
                "text_length"
            ).isNotNull()

            &

            F.col(
                "text_repeat_count"
            ).isNotNull()
        )


        signal = (

            (
                F.col(
                    "text_length"
                )
                >=
                min_text_length
            )

            &

            (
                F.col(
                    "text_repeat_count"
                )
                >=
                min_repetitions
            )
        )


        reason = F.concat(

            F.lit(
                "Texto repetido: "
            ),

            F.col(
                "text_repeat_count"
            ).cast(
                "string"
            ),

            F.lit(
                f" veces | longitud mínima: "
                f"{min_text_length}"
            ),

            F.lit(
                f" | repeticiones mínimas: "
                f"{min_repetitions}"
            )
        )


    # ========================================================
    # R004 · New Account High Activity
    # ========================================================

    elif rule_id == "R004":

        max_account_age_days = int(

            params.get(
                "account_age_days",
                30
            )
        )


        min_reviews = int(

            params.get(
                "min_reviews",
                config["threshold"]
            )
        )


        evaluable = (

            F.col(
                "yelping_since"
            ).isNotNull()

            &

            F.col(
                "account_age_days"
            ).isNotNull()

            &

            F.col(
                "user_review_count"
            ).isNotNull()
        )


        signal = (

            F.col(
                "account_age_days"
            ).between(
                0,
                max_account_age_days
            )

            &

            (
                F.col(
                    "user_review_count"
                )
                >=
                min_reviews
            )
        )


        reason = F.concat(

            F.lit(
                "Edad cuenta: "
            ),

            F.col(
                "account_age_days"
            ).cast(
                "string"
            ),

            F.lit(
                " días | Reviews usuario: "
            ),

            F.col(
                "user_review_count"
            ).cast(
                "string"
            ),

            F.lit(
                f" | max edad: "
                f"{max_account_age_days}"
            ),

            F.lit(
                f" | min reviews: "
                f"{min_reviews}"
            )
        )


    # ========================================================
    # R005 · User-Business Concentration
    # ========================================================

    elif rule_id == "R005":

        concentration = float(

            params.get(
                "concentration",
                config["threshold"]
            )
        )


        min_business_reviews = int(

            params.get(
                "min_business_reviews",
                3
            )
        )


        evaluable = (

            F.col(
                "user_business_review_count"
            ).isNotNull()

            &

            F.col(
                "user_business_concentration"
            ).isNotNull()
        )


        signal = (

            (
                F.col(
                    "user_business_review_count"
                )
                >=
                min_business_reviews
            )

            &

            (
                F.col(
                    "user_business_concentration"
                )
                >=
                concentration
            )
        )


        reason = F.concat(

            F.lit(
                "Reviews User-Business: "
            ),

            F.col(
                "user_business_review_count"
            ).cast(
                "string"
            ),

            F.lit(
                " | concentración: "
            ),

            F.round(
                F.col(
                    "user_business_concentration"
                ),
                4
            ).cast(
                "string"
            ),

            F.lit(
                f" | threshold: "
                f"{concentration}"
            )
        )


    # ========================================================
    # R006 · Business Review Spike
    # ========================================================

    elif rule_id == "R006":

        min_daily_reviews = int(

            params.get(
                "min_daily_reviews",
                10
            )
        )


        spike_ratio = float(

            params.get(
                "spike_ratio",
                config["threshold"]
            )
        )


        evaluable = (

            F.col(
                "business_reviews_day"
            ).isNotNull()

            &

            F.col(
                "business_avg_daily_reviews"
            ).isNotNull()

            &

            F.col(
                "business_spike_ratio"
            ).isNotNull()
        )


        signal = (

            (
                F.col(
                    "business_reviews_day"
                )
                >=
                min_daily_reviews
            )

            &

            (
                F.col(
                    "business_spike_ratio"
                )
                >=
                spike_ratio
            )
        )


        reason = F.concat(

            F.lit(
                "Reviews día: "
            ),

            F.col(
                "business_reviews_day"
            ).cast(
                "string"
            ),

            F.lit(
                " | spike ratio: "
            ),

            F.round(
                F.col(
                    "business_spike_ratio"
                ),
                2
            ).cast(
                "string"
            ),

            F.lit(
                f" | min reviews: "
                f"{min_daily_reviews}"
            ),

            F.lit(
                f" | min spike: "
                f"{spike_ratio}"
            )
        )


    else:

        raise Exception(
            f"""
            ❌ Regla no implementada:

            {rule_id}
            """
        )


    # ========================================================
    # RESULTADO ESTÁNDAR
    # ========================================================

    evaluation_result = (

        F.when(
            ~evaluable,
            F.lit(
                "NOT_EVALUABLE"
            )
        )

        .when(
            signal,
            F.lit(
                "SIGNAL"
            )
        )

        .otherwise(
            F.lit(
                "NO_SIGNAL"
            )
        )
    )


    risk_score = (

        F.when(
            evaluable
            &
            signal,

            F.lit(
                weight
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )


    final_reason = (

        F.when(
            ~evaluable,

            F.lit(
                "No evaluable: "
                "features requeridas no disponibles."
            )
        )

        .otherwise(
            reason
        )
    )


    return F.struct(

        F.lit(
            rule_id
        ).alias(
            "rule_id"
        ),

        F.lit(
            rule_version_id
        ).alias(
            "rule_version_id"
        ),

        evaluation_result.alias(
            "evaluation_result"
        ),

        risk_score.alias(
            "risk_score"
        ),

        final_reason.alias(
            "reason"
        )
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 18. Ejecutar reglas activas


# COMMAND ----------

# ============================================================
# 18. CONSTRUIR EXPRESIONES
# ============================================================

rule_structs = [

    build_rule_struct(
        rule_id
    )

    for rule_id
    in sorted(
        active_rule_ids
    )
]


active_rule_count = len(
    rule_structs
)


print(
    f"Reglas ejecutadas: "
    f"{active_rule_count}"
)


# COMMAND ----------

# ============================================================
# 19. EVALUACIÓN
#
# Una sola pasada lógica sobre Feature Table.
# ============================================================

evaluations = (

    features

    .select(

        "review_id",

        F.array(
            *rule_structs
        ).alias(
            "_rule_evaluations"
        )
    )

    .select(

        "review_id",

        F.explode(
            "_rule_evaluations"
        ).alias(
            "rule"
        )
    )

    .select(

        "review_id",

        F.col(
            "rule.rule_id"
        ).alias(
            "rule_id"
        ),

        F.col(
            "rule.rule_version_id"
        ).alias(
            "rule_version_id"
        ),

        F.col(
            "rule.evaluation_result"
        ).alias(
            "evaluation_result"
        ),

        F.col(
            "rule.risk_score"
        ).alias(
            "risk_score"
        ),

        F.col(
            "rule.reason"
        ).alias(
            "reason"
        )
    )
)


# COMMAND ----------

# ============================================================
# 20. METADATA
# ============================================================

evaluations = (

    evaluations

    .withColumn(
        "evaluation_run_id",

        F.lit(
            evaluation_run_id
        )
    )

    .withColumn(
        "evaluation_id",

        F.sha2(

            F.concat_ws(

                "||",

                F.col(
                    "review_id"
                ),

                F.col(
                    "rule_version_id"
                ),

                F.col(
                    "evaluation_run_id"
                )
            ),

            256
        )
    )

    .withColumn(
        "evaluated_at",

        F.current_timestamp()
    )
)


# COMMAND ----------

# ============================================================
# 21. VALIDAR GRANO
#
# Ya NO existe "* 6" hardcodeado.
# ============================================================

expected_count = (

    review_count
    *
    active_rule_count
)


try:

    actual_count = (
        evaluations.count()
    )


    print(
        f"Reviews            : "
        f"{review_count:,}"
    )

    print(
        f"Reglas ACTIVE      : "
        f"{active_rule_count}"
    )

    print(
        f"Esperadas          : "
        f"{expected_count:,}"
    )

    print(
        f"Generadas          : "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise Exception(
            f"""
            ❌ No se cumple el grano esperado.

            Reviews:
            {review_count}

            Reglas:
            {active_rule_count}

            Esperadas:
            {expected_count}

            Generadas:
            {actual_count}
            """
        )


    print(
        "✔ Grano Review × Rule Version × Run validado."
    )


    # ========================================================
    # 22. PERSISTENCIA
    # ========================================================

    if not spark.catalog.tableExists(
        EVALUATION_TABLE
    ):

        (
            evaluations

            .write

            .format("delta")

            .mode("overwrite")

            .saveAsTable(
                EVALUATION_TABLE
            )
        )

    else:

        (
            evaluations

            .write

            .format("delta")

            .mode("append")

            .saveAsTable(
                EVALUATION_TABLE
            )
        )


    print(
        "✔ slv_review_evaluation persistida."
    )


    # ========================================================
    # 23. FINALIZAR RUN
    # ========================================================

    update_run_status(
        "SUCCESS"
    )


    print(
        "✔ Evaluation Run finalizado: SUCCESS."
    )


except Exception as exc:

    # --------------------------------------------------------
    # Si algo falla después de registrar el Run,
    # lo dejamos explícitamente como FAILED.
    # --------------------------------------------------------

    try:

        update_run_status(
            "FAILED"
        )

    except Exception as update_exc:

        print(
            "⚠ No fue posible actualizar "
            "evaluation_run a FAILED."
        )

        print(
            update_exc
        )


    print()
    print(
        "❌ TRUST & SAFETY FAILED"
    )

    print(
        str(
            exc
        )
    )

    raise


# COMMAND ----------

# MAGIC %md
# MAGIC # 24. Resultados por regla


# COMMAND ----------

display(

    evaluations

    .groupBy(
        "rule_id",
        "evaluation_result"
    )

    .count()

    .orderBy(
        "rule_id",
        "evaluation_result"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 25. Tasa de activación por regla


# COMMAND ----------

rule_statistics = (

    evaluations

    .groupBy(
        "rule_id"
    )

    .agg(

        F.count("*")
        .alias(
            "reviews_evaluated"
        ),

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "SIGNAL",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "signals"
        ),

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NOT_EVALUABLE",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "not_evaluable"
        )
    )

    .withColumn(
        "signal_rate_pct",

        F.round(

            F.col(
                "signals"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            4
        )
    )

    .orderBy(
        "rule_id"
    )
)


display(
    rule_statistics
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 26. Señales detectadas
# MAGIC
# MAGIC Este resultado sigue siendo una evaluación por regla.
# MAGIC
# MAGIC La consolidación por Review ocurre posteriormente en:
# MAGIC
# MAGIC `05_risk_consolidation`.


# COMMAND ----------

display(

    evaluations

    .filter(
        F.col(
            "evaluation_result"
        )
        ==
        "SIGNAL"
    )

    .select(

        "review_id",
        "rule_id",
        "rule_version_id",

        "risk_score",
        "reason",

        "evaluation_run_id"
    )

    .orderBy(
        "rule_id",
        "review_id"
    )

    .limit(100)
)


# COMMAND ----------

# ============================================================
# 27. RESUMEN FINAL
# ============================================================

rule_summary = (

    evaluations

    .groupBy(
        "evaluation_result"
    )

    .count()
)


signal_evaluations = (

    rule_summary

    .filter(
        F.col(
            "evaluation_result"
        )
        ==
        "SIGNAL"
    )

    .select(
        "count"
    )

    .first()
)


signal_evaluation_count = (

    signal_evaluations["count"]

    if signal_evaluations

    else 0
)


not_evaluable_row = (

    rule_summary

    .filter(
        F.col(
            "evaluation_result"
        )
        ==
        "NOT_EVALUABLE"
    )

    .select(
        "count"
    )

    .first()
)


not_evaluable_count = (

    not_evaluable_row["count"]

    if not_evaluable_row

    else 0
)


print()
print("=" * 72)
print("TRUST & SAFETY FINALIZADO")
print("=" * 72)

print(
    f"Evaluation Run        : "
    f"{evaluation_run_id}"
)

print(
    f"Feature Version       : "
    f"{dataset_version}"
)

print(
    f"Reviews evaluadas     : "
    f"{review_count:,}"
)

print(
    f"Reglas ACTIVE         : "
    f"{active_rule_count}"
)

print(
    f"Evaluaciones          : "
    f"{actual_count:,}"
)

print(
    f"Signals por regla     : "
    f"{signal_evaluation_count:,}"
)

print(
    f"Not evaluable         : "
    f"{not_evaluable_count:,}"
)

print(
    f"Output                : "
    f"{EVALUATION_TABLE}"
)

print()
print(
    "Siguiente paso:"
)

print(
    "05_risk_consolidation"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done
# MAGIC
# MAGIC - [ ] Quality Gate Silver superado.
# MAGIC - [ ] Se utiliza el catálogo gobernado T&S.
# MAGIC - [ ] Solo se ejecutan reglas ACTIVE.
# MAGIC - [ ] Las reglas DRAFT no participan.
# MAGIC - [ ] Thresholds provienen de `evaluation_rule_version`.
# MAGIC - [ ] Pesos provienen de `evaluation_rule_version`.
# MAGIC - [ ] No se usa `cache()` / `persist()`.
# MAGIC - [ ] El número de reglas no está hardcodeado.
# MAGIC - [ ] Se cumple Review × Rule Version × Evaluation Run.
# MAGIC - [ ] `evaluation_run` termina SUCCESS o FAILED.
# MAGIC - [ ] No se calcula Risk Consolidation aquí.
# MAGIC - [ ] No se etiqueta fraude confirmado.